# HAR Stage 20A — FIXED fast & resumable notebook (v3; no Google Drive)

**Important fix:** The traceback containing `HAR_Stage20A_ConditionalNull_NoDrive.py` came from the **old slow** runner. This notebook explicitly launches `HAR_Stage20A_Fast_Resumable.py` and captures the full Python traceback if anything fails.

- 152 provisional HAR pairs; six JASPAR 2024 motifs; 4,999 permutations per HAR; one HAR-level six-TF maximum statistic and BH across 152 HARs.
- Original-author genomic HAR identity remains provisional; no FIMO-calibrated p-values, experimentally validated TF binding, or HAR–gene causality is asserted.
- This version retains automatic per-HAR checkpoint files. On an error it creates `HAR_Stage20A_Diagnostics_and_Partial.zip` with an execution log and any partial CSV output.
- **Run cells 1–4 in order.** If the old notebook is still open, switch to this one rather than editing the old slow notebook.

**Resuming saved CSVs:** You can upload `S1_HAR_152_SixTF_FamilyWise_ConditionalNull.csv` and `S2_912_Pairwise_Aligned_TF_Descriptive.csv` into `/content/HAR_Stage20A_FullScreen_Results/` *before* running Cell 4. Only results from the same 4,999-permutation protocol should be combined, and retain provenance of the earlier runner/seed.


In [ ]:
# Step 1 — Reconstruct pinned research inputs and scripts (no Drive)
import io,base64,zipfile,hashlib
from pathlib import Path
root=Path('/content') if Path('/content').exists() else Path.cwd()
PACK="""..."""
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACK))) as z:
    for name in z.namelist():
        file=root/name
        content=z.read(name)
        file.write_bytes(content)
        print(name, 'SHA256:',hashlib.sha256(content).hexdigest()[:16])
print('Original data and optimized scripts ready. Existing partial result files were NOT deleted.')


In [ ]:
# Step 2 — Install required dependencies only if missing
import importlib.util, sys, subprocess
requirements=[('numba','numba>=0.59'),('Bio','biopython>=1.84'),('numpy','numpy>=1.24'),('requests','requests>=2.30')]
missing=[pkg for module,pkg in requirements if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
print('Required Python libraries available.')


In [ ]:
# Step 3 — Verify correct runner and all embedded input files before execution
from pathlib import Path
import sys, py_compile, csv
root=Path('/content') if Path('/content').exists() else Path.cwd()
required=[
  'HAR_Stage20A_Fast_Resumable.py',
  'HAR_Stage20A_ConditionalNull_NoDrive.py',
  'STAGE20A_152_PROVISIONAL_HUMAN.fa',
  'STAGE20A_152_PROVISIONAL_CHIMP.fa',
  'STAGE20A_152_ALIGNMENT_QC.csv',
]
for name in required:
    f=root/name
    if not f.exists() or f.stat().st_size==0:
        raise FileNotFoundError(f'Missing required file: {f} — run Step 1 first')
    print(f'OK  {name} ({f.stat().st_size:,} bytes)')
for name in required[:2]:
    py_compile.compile(str(root/name), doraise=True)
from Bio import SeqIO
with (root/'STAGE20A_152_PROVISIONAL_HUMAN.fa').open() as f: h={x.id.split('|')[0] for x in SeqIO.parse(f,'fasta')}
with (root/'STAGE20A_152_PROVISIONAL_CHIMP.fa').open() as f: c={x.id.split('|')[0] for x in SeqIO.parse(f,'fasta')}
assert len(h)==len(c)==152 and h==c, f'Unexpected FASTA mapping: human={len(h)}, chimp={len(c)}'
print('PASS: 152 paired ortholog sequence labels')
print('Correct optimized executable:',root/'HAR_Stage20A_Fast_Resumable.py')


In [ ]:
# Step 4 — RUN THE FAST VERSION with full error logging and automatic checkpoint rescue
import os, sys, subprocess, zipfile, traceback
from pathlib import Path
from collections import deque
root=Path('/content') if Path('/content').exists() else Path.cwd()
output=root/'HAR_Stage20A_FullScreen_Results'
output.mkdir(parents=True,exist_ok=True)
runner=root/'HAR_Stage20A_Fast_Resumable.py'  # IMPORTANT: not old slow ConditionalNull runner
if not runner.exists():
    raise FileNotFoundError(f'Fast script missing: {runner}; run Step 1 first')
env=os.environ.copy()
env['HAR_N_PERM']='4999'
env['HAR_CHECKPOINT']='1'
env['PYTHONUNBUFFERED']='1'
log=output/'STAGE20A_execution_debug.log'
print('Launching',runner.name,'|',env['HAR_N_PERM'],'permutations/locus',flush=True)
recent=deque(maxlen=60)
with log.open('a',encoding='utf-8') as fp:
    fp.write('\n--- NEW EXECUTION: '+runner.name+' ---\n')
    fp.flush()
    proc=subprocess.Popen([sys.executable,'-u',str(runner)],cwd=str(root),
                          env=env,stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end='',flush=True)
        fp.write(line)
        fp.flush()
        recent.append(line.rstrip('\n'))
    rc=proc.wait()
if rc:
    print('\nSTAGE 20A FAILED. Actual last traceback lines:',flush=True)
    print('\n'.join(recent),flush=True)
    diagnostics=root/'HAR_Stage20A_Diagnostics_and_Partial.zip'
    with zipfile.ZipFile(diagnostics,'w',zipfile.ZIP_DEFLATED) as z:
        for p in output.glob('*.csv'):
            z.write(p,arcname=p.name)
        z.write(log,arcname=log.name)
        for name in ['HAR_Stage20A_Fast_Resumable.py','HAR_Stage20A_ConditionalNull_NoDrive.py']:
            p=root/name
            if p.exists():z.write(p,arcname='source/'+p.name)
    print('Diagnostic ZIP:',diagnostics,flush=True)
    try:
        from google.colab import files
        files.download(str(diagnostics))
    except ImportError:
        print('Download diagnostics ZIP from the Jupyter/Colab Files sidebar.')
    raise RuntimeError('Stage20A failed. See the ACTUAL traceback above or share HAR_Stage20A_Diagnostics_and_Partial.zip; exit code alone is not diagnostic.')
print('SUCCESS! Full Stage20A results:',root/'HAR_Stage20A_FullScreen_Results.zip')


In [ ]:
# Step 5 — Re-download completed or partial output if browser auto-download was blocked
from pathlib import Path
import zipfile
root=Path('/content') if Path('/content').exists() else Path.cwd()
full=root/'HAR_Stage20A_FullScreen_Results.zip'
partial=root/'HAR_Stage20A_Partial_Checkpoint.zip'
folder=root/'HAR_Stage20A_FullScreen_Results'
if full.is_file():
    target=full
elif folder.is_dir() and list(folder.glob('S*.csv')):
    with zipfile.ZipFile(partial,'w',zipfile.ZIP_DEFLATED) as z:
        for f in folder.glob('S*.csv'):z.write(f,arcname=f.name)
        log=folder/'STAGE20A_execution_debug.log'
        if log.is_file():z.write(log,arcname=log.name)
    target=partial
else:
    raise FileNotFoundError('No completed or partial results exist. Run Step 4 and inspect printed traceback if it fails.')
print('Download:',target)
try:
    from google.colab import files
    files.download(str(target))
except ImportError:
    print('Use local Jupyter file browser to download:',target)
